# Notebook 10 — Extractive QA Evaluation

## Public evaluation summary

This notebook documents and verifies the published aggregate results
of the extractive question-answering endpoint `/ask`.

The private evaluation used:

- 40 answerable Greek-language questions
- 32 unique news articles
- A frozen QA evaluation dataset
- 40 saved API predictions
- Retrieval with `k=10`
- A semantic review of 15 flagged cases

**Publication scope:** This public notebook reads aggregate reports.
It does not contain raw news articles, reference answers, exact
evidence excerpts, or individual API responses.

The original full experimental notebook and underlying artifacts
are retained separately. Re-running inference requires those
artifacts and the pinned API environment.

Source: `results/qa_evaluation_v1/`


## 1. Load published evaluation artifacts

The notebook runs on CPU and does not download models.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/Greek-News-NLP-API"),
]

repo = next(
    (
        p for p in candidates
        if (
            p / "results/qa_evaluation_v1/"
                "ask_final_metrics_40.csv"
        ).is_file()
    ),
    None
)

assert repo is not None, (
    "Run this notebook from a clone of Greek-News-NLP-API."
)

results = repo / "results/qa_evaluation_v1"

metrics = pd.read_csv(
    results / "ask_final_metrics_40.csv"
)

audit = json.loads(
    (
        results / "qa_reproducibility_audit_v1.json"
    ).read_text(encoding="utf-8")
)

conclusion = json.loads(
    (
        results / "ask_benchmark_conclusion_v1.json"
    ).read_text(encoding="utf-8")
)

print("Published QA artifacts loaded successfully.")
print("Evaluation questions:", audit["dataset"]["n_questions"])
print("Unique documents:", audit["dataset"]["n_unique_documents"])


## 2. Automated benchmark results

These metrics describe **source retrieval and evidence extraction**,
not the accuracy of standalone generated answers.

The API returns evidence sentences and sources rather than a
generated natural-language answer.


In [ ]:
assert "all_40" in metrics["group"].values

display(
    metrics[
        [
            "group",
            "n_questions",
            "source_hit@1",
            "source_hit@3",
            "source_hit@5",
            "source_hit@10",
            "evidence_doc_hit",
            "exact_quote_hit",
            "mean_quote_token_recall",
            "latency_mean_s",
        ]
    ].round(4)
)


## 3. Verify published metrics and provenance

The following checks compare the published aggregate CSV with
the final reproducibility audit. This is a consistency check
of published results, **not a fresh evaluation run**.


In [ ]:
assert audit["status"] == "reproducibility_audit_passed"
assert conclusion["status"] == (
    "evaluation_and_flagged_review_complete"
)

assert audit["dataset"]["frozen"] is True
assert audit["dataset"]["n_questions"] == 40
assert audit["dataset"]["n_unique_documents"] == 32

assert (
    conclusion["source_artifacts"]["dataset_sha256"]
    == audit["dataset"]["sha256"]
)

assert (
    conclusion["source_artifacts"]["predictions_sha256"]
    == audit["predictions"]["sha256"]
)

row = metrics.loc[
    metrics["group"] == "all_40"
].iloc[0]

expected = {
    "source_hit@1": 0.95,
    "source_hit@3": 0.95,
    "source_hit@5": 0.975,
    "source_hit@10": 0.975,
    "evidence_doc_hit": 0.975,
    "exact_quote_hit": 0.65,
}

for name, value in expected.items():
    assert abs(float(row[name]) - value) < 1e-8, name

assert abs(
    float(row["mean_quote_token_recall"])
    - audit["automated_metrics"]["mean_quote_token_recall"]
) < 1e-6

print("Published aggregate metrics: VERIFIED")
print("Dataset provenance: VERIFIED")
print("Predictions provenance: VERIFIED")
print("Full inference rerun: NOT PERFORMED")


## 4. Flagged-case semantic review

A targeted review examined the 15 cases flagged by retrieval
or lexical evidence checks.

The assistant-proposed judgments were approved by the project
author. This was **not an independent blind human study**.

The 25 unflagged questions were not separately adjudicated,
so the results below must not be interpreted as full-dataset
answer accuracy.


In [ ]:
counts = audit["flagged_review"]

review_table = pd.DataFrame([
    {"Judgment": "Correct", "Cases": counts["correct"]},
    {"Judgment": "Partial", "Cases": counts["partial"]},
    {"Judgment": "Incorrect", "Cases": counts["incorrect"]},
])

assert review_table["Cases"].sum() == 15

display(review_table)

print(
    "Unflagged cases not semantically adjudicated:",
    counts["n_unflagged_not_semantically_reviewed"]
)


## 5. Findings and limitations

**Main findings**

- Source Hit@1: 95.00% (38/40)
- Source Hit@10: 97.50% (39/40)
- Evidence Document Hit: 97.50% (39/40)
- Exact Evidence Quote Match: 65.00% (26/40)

The main observed weakness is evidence sentence selection:
retrieving the correct article does not guarantee that the
selected sentences contain the information required to
answer the question.

**Limitations**

1. The benchmark covers answerable questions only.
2. Forty questions originate from 32 unique documents.
3. Eight documents contribute a second question.
4. Source Hit is not equivalent to final answer accuracy.
5. Only the 15 flagged cases received semantic judgments.
6. No unanswerable or abstention evaluation was performed.
7. The public notebook verifies stored aggregates; it does
   not reproduce inference without the private evaluation
   artifacts.

For further information, see
[`docs/qa_evaluation.md`](../docs/qa_evaluation.md).
